In [1]:
import torch
import matplotlib.pyplot as plt
import os
import time
import numpy as np
import pandas as pd
import random
from matplotlib.patches import Rectangle
import tifffile
import h5py
from torch.utils.data import DataLoader
import imageio.v2 as imageio
from scipy.ndimage import gaussian_filter
from matplotlib.patches import Patch

import importlib as imp
import sys
sys.path.append(os.path.join(os.path.split(os.getcwd())[0], 'scripts'))

import yolo_tiles
imp.reload(yolo_tiles)
from yolo_tiles import img_to_tiles, apply_model_to_tiles, load_test_image, preprocess, tileDataset, remove_bbox_in_overlap

import yolo_help
imp.reload(yolo_help)
from yolo_help import bbox_to_rectangles, imshow, convert_data, Net, get_best_bounding_box_per_cell

import yolo_post_help
imp.reload(yolo_post_help)
from yolo_post_help import postprocess, bb_to_rec, NMS

import yolo_help_3D
imp.reload(yolo_help_3D)
from yolo_help_3D import apply_model_to_orthogonal_slices, gen_3D_GT, recon_down, vol_to_mip_rgb, orthogonal_to_3D, vol_to_mip

import yolo_avg_prec
imp.reload(yolo_avg_prec)
from yolo_avg_prec import get_AP, gt_bbox_to_pred_bbox_format, IOU_3D, yolo_output_to_list

import yolo_3D_pipeline_help
imp.reload(yolo_3D_pipeline_help)
from yolo_3D_pipeline_help import apply_2D_yolo_to_slices, stitch_slices_into_cube

import nibabel as nib
from csbdeep.utils import normalize

from stardist.models import StarDist3D
from cellpose import models

pd.set_option('display.max_rows', 500)

sys.path.append('/home/abenneck/equivariance_model')
import moment_kernels as mk
imp.reload(mk)

2026-11-17 14:01:50.050121: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-11-17 14:01:50.057065: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1794952910.065625 3031105 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1794952910.068215 3031105 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1794952910.074561 3031105 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

<module 'moment_kernels' from '/home/abenneck/equivariance_model/moment_kernels.py'>

In [2]:
def voxel_to_um(voxel_dim, val, val_pos):
    if val_pos == 0:
        return val * voxel_dim[0]
    elif val_pos == 1:
        return val * voxel_dim[1]
    elif val_pos == 2:
        return val * voxel_dim[2]
    else:
        raise ValueError('val_pos must be from [0,1,2]')

In [ ]:
indir = '/home/abenneck/mounts/bmaproot/nafs/shattuck/RodentToolsData/ZW-DT-1-P56-1/yolo_3D_outputs_v04_NMS/'
conf_thresh = 0.3
outdir = f'/home/abenneck/misc_outputs/cell_counting/Ex_488_Em_525_NMS/conf_{str(conf_thresh).replace('.','_')}_origin0'
next_idx = 0
voxel_dim = [2, 1.866, 1.866]
verbose = True

for idx, fname in enumerate(os.listdir(indir)):
    
    if idx < next_idx:
        continue    

    print(f'Starting {fname} ({idx}/{len(os.listdir(indir))}) . . .')
    start = time.time()
    
    data = np.load(os.path.join(indir, fname))
    if len(np.unique(data[...,6])) == 1:
        print('Skipping empty cube . . .\n')
        continue

    out_data = list()
    count = 0
    
    cube_slice = int(fname[1:5])
    cube_row   = int(fname[7:11])
    cube_col   = int(fname[13:17])

    for i in range(data.shape[0]):
        for j in range(data.shape[1]):
            for k in range(data.shape[2]):
                pred = data[i,j,k]

                conf = pred[6]
                
                if conf < conf_thresh:
                    continue

                if verbose:
                    # print(f'Adding cell {i*data.shape[0] + j*data.shape[1] + k*data.shape[2]:_} / {data.shape[0]*data.shape[1]*data.shape[2]:_}')
                    count += 1

                slice_min = voxel_to_um(voxel_dim, pred[0], 0)
                slice_max = voxel_to_um(voxel_dim, pred[1], 0)
                row_min = voxel_to_um(voxel_dim, pred[2], 1)
                row_max = voxel_to_um(voxel_dim, pred[3], 1)
                col_min = voxel_to_um(voxel_dim, pred[4], 2)
                col_max = voxel_to_um(voxel_dim, pred[5], 2)
                
                cl0 = pred[7]
                cl1 = pred[8]
                cl2 = pred[9]

                c_slice = (slice_min + slice_max) / 2
                c_row = (row_min + row_max) / 2
                c_col = (col_min + col_max) / 2

                l_slice = np.abs(slice_max - slice_min)
                l_row = np.abs(row_max - row_min)
                l_col = np.abs(col_max - col_min)
                
                out_data.append([cube_slice, cube_row, cube_col, c_slice, c_row, c_col, l_slice, l_row, l_col, conf, cl0, cl1, cl2])            

    out_fname = fname[:17] + f'_counts_n{count}.npy'
    np.save(os.path.join(outdir, out_fname), np.array(out_data))
    
    print(f'Finished appending {count} cells in {time.time() - start:.2f}s\n')

    # break

In [49]:
indir = '/home/abenneck/misc_outputs/cell_counting/Ex_488_Em_525_NMS/conf_0_3'

# out_fname = '/home/abenneck/misc_outputs/cell_counting/Ex_488_Em_525_NMS/conf_0_3.pkl'
# columns = ["cube_slice", "cube_row", "cube_col", "c_slice", "c_row", "c_col", "l_slice", "l_row", "l_col", "conf", "cl0", "cl1", "cl2"]

out_fname = '/home/abenneck/misc_outputs/cell_counting/Ex_488_Em_525_NMS/conf_0_3.h5'
columns = ["cube_slice", "cube_row", "cube_col", "c_slice", "c_row", "c_col", "l_slice", "l_row", "l_col"]

next_idx = 3
        
for idx, fname in enumerate(os.listdir(indir)):

    if idx < next_idx:
        continue

    print(f'Starting {idx}/{len(os.listdir(indir))} . . .')

    # out_df = pd.read_pickle(out_fname)
    if idx == 0 and 'h5' in out_fname:
        out_df = pd.DataFrame(columns = ["cube_slice", "cube_row", "cube_col", "c_slice", "c_row", "c_col", "l_slice", "l_row", "l_col"], dtype = int)
        # out_df.to_hdf(out_fname, key="table", mode="a", format="table")
    else:
        out_df = pd.read_hdf(out_fname, key='table', mode="a")

    data = np.load(os.path.join(indir, fname))
    if data.shape == (0,):
        print(f'{fname} is empty, skipping it . . .')
        continue
    data_ = [[round(i) for i in row] for row in data[:,:9]]
    new_df = pd.DataFrame(data_, columns=columns, dtype=int)

    out_df = pd.concat([out_df, new_df])
    # out_df.to_pickle(out_fname, protocol=5)
    out_df.to_hdf(out_fname, key='table', mode="a")

Starting 3/187 . . .
Starting 4/187 . . .
Starting 5/187 . . .
Starting 6/187 . . .
Starting 7/187 . . .
Starting 8/187 . . .
Starting 9/187 . . .
Starting 10/187 . . .
Starting 11/187 . . .
Starting 12/187 . . .
Starting 13/187 . . .
Starting 14/187 . . .
Starting 15/187 . . .
Starting 16/187 . . .
Starting 17/187 . . .
Starting 18/187 . . .
Starting 19/187 . . .
Starting 20/187 . . .
Starting 21/187 . . .
Starting 22/187 . . .
Starting 23/187 . . .
Starting 24/187 . . .
Starting 25/187 . . .
Starting 26/187 . . .
Starting 27/187 . . .
Starting 28/187 . . .
Starting 29/187 . . .
Starting 30/187 . . .
Starting 31/187 . . .
Starting 32/187 . . .
Starting 33/187 . . .
Starting 34/187 . . .
Starting 35/187 . . .
Starting 36/187 . . .
Starting 37/187 . . .
Starting 38/187 . . .
Starting 39/187 . . .
Starting 40/187 . . .
Starting 41/187 . . .
Starting 42/187 . . .
Starting 43/187 . . .
Starting 44/187 . . .
Starting 45/187 . . .
Starting 46/187 . . .
Starting 47/187 . . .
Starting 48/187 .

In [6]:
# # Convert the .pkl file into an .h5 file (TOO MUCH MEMORY)
# in_fname = '/home/abenneck/misc_outputs/cell_counting/Ex_488_Em_525_NMS/conf_0_3.pkl'
# out_fname = '/home/abenneck/misc_outputs/cell_counting/Ex_488_Em_525_NMS/conf_0_3.h5'

# data = pd.DataFrame([])
# data.to_hdf(out_fname, key="table", mode="a", format="table")

In [43]:
# Intialize the DataFrame + Save as .pkl or .h5 file
# df = pd.DataFrame(columns = ["cube_slice", "cube_row", "cube_col", "c_slice", "c_row", "c_col", "l_slice", "l_row", "l_col", "conf", "cl0", "cl1", "cl2"])
df = pd.DataFrame(columns = ["cube_slice", "cube_row", "cube_col", "c_slice", "c_row", "c_col", "l_slice", "l_row", "l_col"], dtype = int)

# out_fname = '/home/abenneck/misc_outputs/cell_counting/Ex_488_Em_525_NMS/conf_0_3.pkl'
# df.to_pickle(out_fname, protocol=5)

out_fname = '/home/abenneck/misc_outputs/cell_counting/Ex_488_Em_525_NMS/conf_0_3.h5'
df.to_hdf(out_fname, key="table", mode="a", format="table")

In [3]:
out_fname = '/home/abenneck/misc_outputs/cell_counting/Ex_488_Em_525_NMS/conf_0_3.h5'
out_df_h5 = pd.read_hdf(out_fname, key="table", mode="a")

out_df_h5.head()

,cube_slice,cube_row,cube_col,c_slice,c_row,c_col,l_slice,l_row,l_col
0,2,1,2,4100,1908,4495,20,16,19
1,2,1,2,4102,1910,4630,10,23,12
2,2,1,2,4101,1908,4774,16,14,15
3,2,1,2,4097,1910,4821,11,16,9
4,2,1,2,4099,1911,4823,19,16,10


In [11]:
out_fname = '/home/abenneck/misc_outputs/cell_counting/Ex_488_Em_525_NMS/conf_0_3.pkl'
out_df_pkl = pd.read_pickle(out_fname)

out_df_pkl.head()

,cube_slice,cube_row,cube_col,c_slice,c_row,c_col,l_slice,l_row,l_col,conf,cl0,cl1,cl2
0,2.0,1.0,2.0,4099.824219,1908.150391,4495.180664,20.001465,16.392944,18.506348,0.469228,-1.39782,-0.289171,0.748363
1,2.0,1.0,2.0,4102.341309,1909.708496,4630.251465,9.629883,22.632568,12.150391,0.352865,-1.157141,-0.492396,0.330309
2,2.0,1.0,2.0,4101.232422,1908.155029,4774.298828,15.681152,14.338501,14.991699,0.651453,-0.359325,-0.631465,-0.839321
3,2.0,1.0,2.0,4097.318359,1910.303467,4821.268555,10.615723,16.286011,8.728027,0.447413,-0.524833,-0.114612,0.026317
4,2.0,1.0,2.0,4098.69873,1911.343018,4822.849609,19.353516,16.089478,10.103516,0.309374,-0.453654,-0.612545,-0.022999


# Assign region labels to cells based on cell center

In [54]:
# Load the original hdf5 file (Voxel => um, but no origin shift)
out_fname = '/home/abenneck/misc_outputs/cell_counting/Ex_488_Em_525_NMS/conf_0_3.h5'
out_df_h5 = pd.read_hdf(out_fname, key="table", mode="a")

# # Change coordinate space to have origin at the center instead of upper left hand corner
# out_df_h5['c_slice'] = out_df_h5['c_slice'] - (ntif[0]-1)/2*dtif[0]
# out_df_h5['c_row'] = out_df_h5['c_row'] - (ntif[1]-1)/2*dtif[1]
# out_df_h5['c_col'] = out_df_h5['c_col'] - (ntif[2]-1)/2*dtif[2]
# out_fname = '/home/abenneck/misc_outputs/cell_counting/Ex_488_Em_525_NMS/conf_0_3_origin_center.h5'
# out_df_h5.to_hdf(out_fname, key="table", mode="a", format="table")

# # Load the adjusted hdf5 file (Voxel => um + origin shift)
# out_fname = '/home/abenneck/misc_outputs/cell_counting/Ex_488_Em_525_NMS/conf_0_3_origin_center.h5'
# out_fname = '/home/abenneck/mounts/bmaproot/nafs/shattuck/RodentToolsData/ZW-DT-1-P56-1/yolo_3D_outputs_v04_NMS/conf_0_3_origin_center.h5'
# out_df_h5 = pd.read_hdf(out_fname, key="table", mode="a")
out_df_h5.head()

,cube_slice,cube_row,cube_col,c_slice,c_row,c_col,l_slice,l_row,l_col
0,2,1,2,4100,1908,4495,20,16,19
1,2,1,2,4102,1910,4630,10,23,12
2,2,1,2,4101,1908,4774,16,14,15
3,2,1,2,4097,1910,4821,11,16,9
4,2,1,2,4099,1911,4823,19,16,10


In [59]:
print(np.min(out_df_h5['c_slice'])/2, np.max(out_df_h5['c_slice'])/2)
print(np.min(out_df_h5['c_row'])/1.866, np.max(out_df_h5['c_row'])/1.866)
print(np.min(out_df_h5['c_col'])/1.866, np.max(out_df_h5['c_col'])/1.866)

69.0 3541.5
94.85530546623794 8497.320471596999
527.8670953912111 6906.752411575562


In [13]:
registration_output_dir = '/home/abenneck/mounts/bmaproot/nafs/shattuck/RodentToolsData/ZW-DT-1-P56-1/registration_outputs'
label_file = os.path.join(registration_output_dir,'transformed_atlas_labels.nii')
vol = nib.load(label_file)
L = np.array(vol.dataobj)

o = vol.affine[:3,-1]
d = np.diag(vol.affine[:3,:3])
n = np.array(vol.shape[-3:])
xL = [np.arange(ni)*di + oi for ni,di,oi in zip(n,d,o)]

dtif = np.array([2.0, 1.866, 1.866]) # todo, add more decimal places if we have them
otif = np.array([0,0,0])
ntif = np.array([3600,8587,7321]) # 3600 slices 8587 rows 7321 slices

1204539731.py (4): __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments. To learn more, see the migration guide https://numpy.org/devdocs/numpy_2_0_migration_guide.html#adapting-to-changes-in-the-copy-keyword


In [29]:
# Convert from microns into voxel label units
c_slice = np.round((out_df_h5['c_slice'] - o[0])/d[0]).astype(int)
c_row   = np.round((out_df_h5['c_row']   - o[1])/d[1]).astype(int)
c_col   = np.round((out_df_h5['c_col']   - o[2])/d[2]).astype(int)

In [30]:
# Evaluate the labels at every cell center
cell_anatomical_labels = L[(np.zeros(c_slice.shape[0],dtype=int), c_slice, c_row, c_col)]
out_df_h5['Region ID'] = cell_anatomical_labels

array([ 74, 142, 142, ...,   0,   0,   0],
      shape=(114489273,), dtype=uint16)

In [34]:
out_df_h5.to_hdf(out_fname, key="table", mode="a", format="table")

# Data Exploration

In [25]:
fname = '/home/abenneck/misc_outputs/cell_counting/Ex_488_Em_525_NMS/conf_0_3_origin_center.h5'
data = pd.read_hdf(fname, key="table", mode="a")

In [23]:
out_df_h5['cube_slice'] = out_df_h5['cube_slice'].astype(np.uint8)
out_df_h5['cube_row'] = out_df_h5['cube_row'].astype(np.uint8)
out_df_h5['cube_col'] = out_df_h5['cube_col'].astype(np.uint8)

out_df_h5['c_slice'] = out_df_h5['c_slice'].astype(np.int16)
out_df_h5['c_row'] = out_df_h5['c_row'].astype(np.int16)
out_df_h5['c_col'] = out_df_h5['c_col'].astype(np.int16)

out_df_h5['l_slice'] = out_df_h5['l_slice'].astype(np.uint8)
out_df_h5['l_row'] = out_df_h5['l_row'].astype(np.uint8)
out_df_h5['l_col'] = out_df_h5['l_col'].astype(np.uint8)
out_df_h5['Region ID'] = out_df_h5['Region ID'].astype(np.uint16)

data = out_df_h5.to_hdf(fname, key="table", mode="a", index = False)

In [7]:
vals, counts = np.unique(data['Region ID'], return_counts=True)

[print(v,c) for v, c in zip(vals,counts)]

0 3639291
2 172031
3 883531
4 464032
6 229242
7 121955
8 703817
9 12347
10 7951
11 47332
12 21085
13 4818
14 95389
15 421397
16 92664
18 3100995
19 77583
20 96942
21 25115
22 10208
23 159089
24 37500
25 6417
26 1081
27 136219
28 239523
29 815442
30 150261
31 35444
32 425286
33 126335
34 39629
35 378043
36 87980
37 7997
38 198559
39 67348
40 41891
41 260985
42 3604710
43 322407
44 19952
45 29260
46 755707
47 267434
48 38683
49 35162
50 598072
51 217552
52 184409
53 23238
55 4856
59 58534
60 17148
61 17304
62 2013
63 139851
64 2964
65 5504
67 387717
68 73165
72 2751
74 1515720
76 10412
78 126286
80 57188
81 28671
82 81544
83 82143
84 90961
86 155394
87 183038
88 32875
89 112430
90 27128
91 29761
92 22892
93 1315121
96 498037
97 36980
99 71338
100 130933
104 86452
105 44790
106 626177
107 12456
108 14452
110 2632
111 42050
112 74502
113 26512
114 8370
115 17778
117 67675
118 33366
119 76370
120 20940
122 351681
123 276402
124 447239
125 17861
126 12367
129 105767
130 19990
131 1079518
134

[None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,

# Append other objects to the hdf5 file

In [12]:
# out_fname = '/home/abenneck/mounts/bmaproot/nafs/shattuck/RodentToolsData/ZW-DT-1-P56-1/yolo_3D_outputs_v04_NMS/conf_0_3_origin_center.h5'
# out_df_h5 = pd.read_hdf(out_fname, key="data", mode="a")

# # Remove the cube columns and add the feature columns as float32
# # out_df_h5 = out_df_h5.drop(columns=["cube_slice","cube_row","cube_col"])
# # out_df_h5['cl0'] = out_df_pkl['cl0'].astype(np.float32)
# # out_df_h5['cl1'] = out_df_pkl['cl1'].astype(np.float32)

# out_df_h5.to_hdf(out_fname, key="data", mode="a", format="table", index = False)

In [52]:
# # Initialize the hdf5 file by defining a few new objects

# fname = '/home/abenneck/mounts/bmaproot/nafs/shattuck/RodentToolsData/ZW-DT-1-P56-1/yolo_3D_outputs_v04_NMS/conf_0_3_origin_center.h5'
# with h5py.File(fname, "a") as f:
#     properties = f.create_group("properties")

#     properties.create_dataset("voxel_dim", data=[2.0, 1.866, 1.866])
#     properties.create_dataset("data_dim", data=[3600,8587,7321])
    
#     out_df_h5.to_hdf(fname, key="data", mode="a", format="table", index = False)

In [21]:
# # # Change units from microns back to pixels
# # # NOTE: To evaluate a cell's location in the atlas, coords must be in units of microns with an origin center,
# # # BUT we want to store the cell's location in terms of pixels (i.e. units of the original input data)

# # in_fname  = '/home/abenneck/misc_outputs/cell_counting/Ex_488_Em_525_NMS/conf_0_3.h5'
# in_fname  = '/home/abenneck/misc_outputs/cell_counting/Ex_488_Em_525_NMS/conf_0_3_origin_center.h5'
# out_fname = '/home/abenneck/mounts/bmaproot/nafs/shattuck/RodentToolsData/ZW-DT-1-P56-1/yolo_3D_outputs_v04_NMS/conf_0_3_origin_center.h5'

# data_in = pd.read_hdf(in_fname, key="table", mode="a")
# data_out = pd.read_hdf(out_fname, key="data", mode="a")

# # data_out["c_slice"] = data_in["c_slice"] / 2
# # data_out["c_slice"] = data_out["c_slice"].astype(np.uint16)
# # data_out["c_row"] = data_in["c_row"] / 1.866
# # data_out["c_row"] = data_out["c_row"].astype(np.uint16)
# # data_out["c_col"] = data_in["c_col"] / 1.866
# # data_out["c_col"] = data_out["c_col"].astype(np.uint16)

# # data_out["l_slice"] = (data_in["l_slice"] / 2).astype(np.uint8)
# # data_out["l_row"] = (data_in["l_row"] / 1.866).astype(np.uint8)
# # data_out["l_col"] = (data_in["l_col"] / 1.866).astype(np.uint8)

# # data_out["Region ID"] = data_in["Region ID"].astype(np.uint16)

# data_out.to_hdf(out_fname, key="data", mode="a", format="table", index = False)

In [22]:
fname = '/home/abenneck/mounts/bmaproot/nafs/shattuck/RodentToolsData/ZW-DT-1-P56-1/yolo_3D_outputs_v04_NMS/conf_0_3_origin_center.h5'
with h5py.File(fname, "a") as f:
    
    def print_structure(name, obj):
        print(name, type(obj))
    
    f.visititems(print_structure)
    print()

    dtif = np.array(f['properties/voxel_dim'])
    print(f'dtif: {dtif}')

    ntif = np.array(f['properties/data_dim'])
    print(f'ntif: {ntif}')

    data = pd.read_hdf(fname, key='data', mode='a')
    print(data.head())

data <class 'h5py._hl.group.Group'>
data/table <class 'h5py._hl.dataset.Dataset'>
properties <class 'h5py._hl.group.Group'>
properties/data_dim <class 'h5py._hl.dataset.Dataset'>
properties/voxel_dim <class 'h5py._hl.dataset.Dataset'>

dtif: [2.    1.866 1.866]
ntif: [3600 8587 7321]
   c_slice  c_row  c_col  l_slice  l_row  l_col       cl0       cl1  Region ID
0     2050   1022   2408       10      8     10 -1.397820 -0.289171         74
1     2051   1023   2481        5     12      6 -1.157141 -0.492396        142
2     2050   1022   2558        8      7      8 -0.359325 -0.631465        142
3     2048   1023   2583        5      8      4 -0.524833 -0.114612        162
4     2049   1024   2584        9      8      5 -0.453654 -0.612545        162


In [26]:
print(f'dataset dimensions: {ntif}')
print(f'slice: [{np.min(data['c_slice'])}, {np.max(data['c_slice'])}]')
print(f'row:   [{np.min(data['c_row'])}, {np.max(data['c_row'])}]')
print(f'col:   [{np.min(data['c_col'])}, {np.max(data['c_col'])}]')

dataset dimensions: [3600 8587 7321]
slice: [69, 3541]
row:   [94, 8497]
col:   [527, 6906]


In [29]:
for col in data.columns:
    print(f'{col:<10} {data[col].dtype}')

c_slice    uint16
c_row      uint16
c_col      uint16
l_slice    uint8
l_row      uint8
l_col      uint8
cl0        float32
cl1        float32
Region ID  uint16
